# 1. Load Scored data into SQLite

In [1]:
import pandas as pd
import sqlite3
import os

FINAL = r"C:\Users\alokhande\Desktop\Project_Daedalus\data\final"
DB_PATH = r"C:\Users\alokhande\Desktop\Project_Daedalus\data\daedalus.db"

df = pd.read_csv(os.path.join(FINAL, "daedalus_scored.csv"))

# Create SQLite database and load data
conn = sqlite3.connect(DB_PATH)
df.to_sql("cities", conn, if_exists="replace", index=False)

print(f"Database created at {DB_PATH}")
print(f"Table 'cities' loaded with {len(df)} rows")

# Verify
cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
print(f"Tables: {cursor.fetchall()}")
cursor.execute("PRAGMA table_info(cities)")
print(f"\nSchema:")
for col in cursor.fetchall():
    print(f"  {col[1]} — {col[2]}")

Database created at C:\Users\alokhande\Desktop\Project_Daedalus\data\daedalus.db
Table 'cities' loaded with 359 rows
Tables: [('cities',)]

Schema:
  city — TEXT
  aqi_value — INTEGER
  aqi_category — TEXT
  pm2.5_aqi_value — INTEGER
  meal_cheap_usd — REAL
  meal_midrange_usd — REAL
  rent_1br_outside_usd — REAL
  rent_3br_city_usd — REAL
  basic_utilities_usd — REAL
  monthly_pass_usd — REAL
  avg_net_salary_usd — REAL
  popcity — REAL
  densitycity — REAL
  congestion2019 — REAL
  crime_index — REAL
  safety_index — REAL
  aqi_score — REAL
  pm25_score — REAL
  air_quality_score — REAL
  cost_index_raw — REAL
  cost_score_absolute — REAL
  monthly_living_cost — REAL
  affordability_ratio — REAL
  cost_score_affordability — REAL
  safety_score — REAL
  density_score_raw — REAL
  congestion_score — REAL
  urban_score — REAL
  livability_score_absolute — REAL
  livability_score_affordability — REAL
  rank_absolute — INTEGER
  rank_affordability — INTEGER
  country — TEXT


# 2. Helper function (write SQL, get a DataFrame back):

In [2]:
def query(sql):
    return pd.read_sql_query(sql, conn)

# 3. Q1: Top 10 most livable cities overall:

In [4]:
result = query("""
    SELECT 
        rank_absolute,
        city      AS city,
        country,
        ROUND(livability_score_absolute, 2) AS livability_score,
        ROUND(air_quality_score, 2)  AS air_quality,
        ROUND(safety_score, 2)       AS safety,
        ROUND(cost_score_absolute, 2) AS cost,
        ROUND(urban_score, 2)        AS urban
    FROM cities
    ORDER BY rank_absolute
    LIMIT 10
""")
print("Q1 — Top 10 most livable cities")
print(result.to_string(index=False))

Q1 — Top 10 most livable cities
 rank_absolute      city      country  livability_score  air_quality  safety  cost  urban
             1   tampere      Finland             90.26        98.13   84.67 87.97  90.86
             2 ljubljana     Slovenia             89.80        97.64   85.36 88.08  88.11
             3 trondheim       Norway             89.53        97.27   89.09 81.58  92.32
             4 stavanger       Norway             89.27        97.94   83.29 83.53  94.50
             5      bern  Switzerland             88.18        95.95   91.02 79.41  88.09
             6 groningen  Netherlands             87.73        94.29   84.94 85.65  86.16
             7   trieste        Italy             87.52        94.95   78.18 88.69  88.17
             8    malaga        Spain             87.25        92.18   78.04 88.27  91.05
             9   antalya       Turkey             87.08        95.32   78.04 94.13  77.52
            10     bursa       Turkey             86.52        90.42

# 4. Q2: Which continent has the highest average livability?

In [5]:
result = query("""
    SELECT
        country,
        COUNT(*)                                    AS city_count,
        ROUND(AVG(livability_score_absolute), 2)   AS avg_livability,
        ROUND(AVG(air_quality_score), 2)            AS avg_air_quality,
        ROUND(AVG(safety_score), 2)                 AS avg_safety,
        ROUND(AVG(cost_score_absolute), 2)          AS avg_cost,
        ROUND(AVG(urban_score), 2)                  AS avg_urban
    FROM cities
    GROUP BY country
    HAVING COUNT(*) >= 3
    ORDER BY avg_livability DESC
    LIMIT 15
""")
print("Q2 — Countries with highest avg livability (min 3 cities)")
print(result.to_string(index=False))

Q2 — Countries with highest avg livability (min 3 cities)
           country  city_count  avg_livability  avg_air_quality  avg_safety  avg_cost  avg_urban
            Poland           4           84.38            93.92       77.11     89.65      73.64
       Netherlands           4           83.99            95.17       77.56     78.00      87.05
            Turkey           5           83.10            91.44       67.24     96.44      72.47
            Turkey           4           81.67            90.64       51.10     98.99      82.68
             Spain           8           81.22            94.07       51.10     91.66      87.16
           Germany          13           80.75            91.70       63.63     84.94      82.19
Russian Federation           5           79.72            94.27       51.10     95.74      73.26
            Poland           5           79.69            92.06       51.10     91.56      82.13
         Australia          10           79.48            94.60      

# 5. Q3: Hidden gems — high livability, low rent:

In [6]:
result = query("""
    SELECT
        city                               AS city,
        country,
        ROUND(livability_score_absolute, 2)        AS livability_score,
        ROUND(rent_1br_outside_usd, 2)             AS rent_usd,
        ROUND(avg_net_salary_usd, 2)               AS avg_salary_usd,
        ROUND(safety_score, 2)                     AS safety_score,
        ROUND(air_quality_score, 2)                AS air_quality_score
    FROM cities
    WHERE livability_score_absolute > 75
      AND rent_1br_outside_usd < 500
    ORDER BY livability_score_absolute DESC
    LIMIT 15
""")
print("Q3 — Hidden gems: high livability + rent under $500/month")
print(result.to_string(index=False))

Q3 — Hidden gems: high livability + rent under $500/month
     city    country  livability_score  rent_usd  avg_salary_usd  safety_score  air_quality_score
  antalya     Turkey             87.08    347.28          327.95         78.04              95.32
    bursa     Turkey             86.52    175.27          331.46         75.28              90.42
   taipei     Taiwan             86.37    441.34         1876.87         94.61              71.50
     baku Azerbaijan             85.90    181.70          405.72         73.34              84.94
   poznan     Poland             85.17    434.74          970.75         76.52              94.37
    izmir     Turkey             84.44    163.04          308.30         72.10              88.45
  dresden    Germany             84.35    465.40         2579.58         72.65              90.42
nuremberg    Germany             83.87    493.50         2992.03         67.27              96.31
   ankara     Turkey             83.63    154.32          37

# 6. Q4: The pollution problem — most livable cities with bad air:

In [7]:
result = query("""
    SELECT
        city                           AS city,
        country,
        aqi_value,
        ROUND(air_quality_score, 2)            AS air_score,
        ROUND(safety_score, 2)                 AS safety_score,
        ROUND(cost_score_affordability, 2)     AS affordability_score,
        ROUND(livability_score_absolute, 2)    AS livability_score
    FROM cities
    WHERE aqi_value > 100
    ORDER BY livability_score_absolute DESC
    LIMIT 10
""")
print("Q4 — Cities with poor air quality that still rank well overall")
print(result.to_string(index=False))

Q4 — Cities with poor air quality that still rank well overall
     city               country  aqi_value  air_score  safety_score  affordability_score  livability_score
   taipei                Taiwan        167      71.50         94.61                94.72             86.37
     doha                 Qatar        164      69.03         95.44                92.37             81.01
  beijing                 China        123      75.30         76.24                89.37             80.44
 helsinki               Finland        162      66.75         81.22                91.69             80.13
 valencia                 Spain        115      77.05         71.55                84.55             79.32
abu dhabi  United Arab Emirates        170      67.72        100.00                94.10             79.12
 shanghai                 China        156      68.61         73.20                89.01             78.66
   riyadh          Saudi Arabia        196      59.29         78.45              

# 7. Safety Vs Affordability Tradeoff

In [9]:
result = query("""
    SELECT
        city                           AS city,
        country,
        ROUND(safety_score, 2)                 AS safety_score,
        ROUND(cost_score_affordability, 2)     AS affordability_score,
        ROUND(avg_net_salary_usd, 2)           AS avg_salary_usd,
        ROUND(crime_index, 2)                  AS crime_index,
        ROUND(livability_score_absolute, 2)    AS livability_score,
        CASE
            WHEN safety_score > 70 AND cost_score_affordability > 70 
                THEN 'Safe AND affordable'
            WHEN safety_score > 70 AND cost_score_affordability <= 70 
                THEN 'Safe but expensive'
            WHEN safety_score <= 70 AND cost_score_affordability > 70 
                THEN 'Affordable but risky'
            ELSE 'Needs improvement'
        END AS category
    FROM cities
    ORDER BY safety_score DESC, cost_score_affordability DESC
    LIMIT 20
""")
print("Q5 — Safety vs affordability tradeoff")
print(result.to_string(index=False))

# Summary counts
summary = query("""
    SELECT
        CASE
            WHEN safety_score > 70 AND cost_score_affordability > 70 
                THEN 'Safe AND affordable'
            WHEN safety_score > 70 AND cost_score_affordability <= 70 
                THEN 'Safe but expensive'
            WHEN safety_score <= 70 AND cost_score_affordability > 70 
                THEN 'Affordable but risky'
            ELSE 'Needs improvement'
        END AS category,
        COUNT(*) as city_count
    FROM cities
    GROUP BY category
    ORDER BY city_count DESC
""")
print("\nCategory breakdown across all 356 cities:")
print(summary.to_string(index=False))

Q5 — Safety vs affordability tradeoff
     city               country  safety_score  affordability_score  avg_salary_usd  crime_index  livability_score            category
abu dhabi  United Arab Emirates        100.00                94.10         3541.40         11.2             79.12 Safe AND affordable
     doha                 Qatar         95.44                92.37         3342.40         14.5             81.01 Safe AND affordable
   taipei                Taiwan         94.61                94.72         1876.87         15.1             86.37 Safe AND affordable
   taipei                Taiwan         94.61                94.72         1876.87         15.1             76.18 Safe AND affordable
  sharjah  United Arab Emirates         93.51                11.36         1681.97         15.9             56.75  Safe but expensive
    dubai  United Arab Emirates         92.82                94.98         4322.14         16.4             71.71 Safe AND affordable
     bern           Swit

# 8.  Q6: Best cities for remote workers (safe + affordable + good air):

In [11]:
result = query("""
    SELECT
        city                           AS city,
        country,
        ROUND(air_quality_score, 2)            AS air_quality,
        ROUND(safety_score, 2)                 AS safety,
        ROUND(cost_score_affordability, 2)     AS affordability,
        ROUND(rent_1br_outside_usd, 2)         AS rent_usd,
        ROUND(livability_score_absolute, 2)    AS livability_score
    FROM cities
    WHERE air_quality_score   > 60
      AND safety_score        > 60
      AND cost_score_affordability > 60
      AND rent_1br_outside_usd < 800
    ORDER BY livability_score_absolute DESC
    LIMIT 15
""")
print("Q6 — Best cities for remote workers")
print("     (air quality > 60, safety > 60, affordability > 60, rent < $800)")
print(result.to_string(index=False))

Q6 — Best cities for remote workers
     (air quality > 60, safety > 60, affordability > 60, rent < $800)
     city         country  air_quality  safety  affordability  rent_usd  livability_score
  tampere         Finland        98.13   84.67          92.68    634.00             90.26
ljubljana        Slovenia        97.64   85.36          81.33    570.38             89.80
groningen     Netherlands        94.29   84.94          86.93    670.77             87.73
  trieste           Italy        94.95   78.18          82.72    555.85             87.52
   malaga           Spain        92.18   78.04          88.03    610.36             87.25
    bursa          Turkey        90.42   75.28          70.19    175.27             86.52
   taipei          Taiwan        71.50   94.61          94.72    441.34             86.37
  vilnius       Lithuania        96.08   77.35          81.58    533.76             85.97
     baku      Azerbaijan        84.94   73.34          75.75    181.70             

# 9. Cell 9 — Close connection and confirm:

In [14]:
conn.close()
print("Database connection closed.")
print(f"\nSQL analysis complete — 6 queries run")
print(f"Database saved at: {DB_PATH}")

Database connection closed.

SQL analysis complete — 6 queries run
Database saved at: C:\Users\alokhande\Desktop\Project_Daedalus\data\daedalus.db
